# SMAP Colab — Simulação + Calibração/Validação
## Semiárido Paraibano · Reservatórios Engenheiro Ávidos e São Gonçalo

**Modelo:** SMAP (*Soil Moisture Accounting Procedure*) — Lopes (1982) · escala mensal
**Dados:** Precipitação (1963–2019) + Q observada **São Gonçalo** (1985–2012)
**Referências:** AESA (2016) · Nash & Sutcliffe (1970) · Moriasi et al. (2007) · Klemeš (1986)

---

### Fluxo do notebook

| # | Células | Conteúdo |
|---|---------|----------|
| 1–4 | Setup | Dependências · Upload · Imports |
| 5 | Constantes | Parâmetros SMAP, ETP, calibração |
| 6–7 | Leitura | `load_precipitation` · `load_observed_flow` |
| 8–10 | Núcleo SMAP | `build_etp_series` · `smap_step` · `run_smap` |
| 11 | Métricas | `nse` · `pbias` · `classify_*` |
| 12 | Calibração | `objective_function` · `calibrate` |
| 13–14 | Saídas | `plot_simulation` · `save_flow_csv` · `plot_calibration` |
| 15 | Q_obs (São Gonçalo) | Carrega vazões observadas — única bacia monitorada |
| 16–17 | São Gonçalo | Calibra (Evolução Diferencial · KGE) → Simula com parâmetros calibrados |
| 18 | Eng. Ávidos | Simula por **transferência** dos parâmetros de São Gonçalo (regionalização) |
| 19 | Download | CSVs e figuras PNG |

> **Bacia monitorada × não monitorada:** apenas **São Gonçalo** possui vazões
> observadas; é nela que o modelo é calibrado. **Engenheiro Ávidos** não tem
> dados fluviométricos — suas vazões são geradas por **regionalização por
> proximidade**, transferindo os parâmetros calibrados em São Gonçalo.

In [ ]:
import importlib, subprocess, sys

for pkg in ['seaborn', 'scipy']:
    if importlib.util.find_spec(pkg) is None:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Dependências OK.')

Dependências OK.


## Upload dos Dados de Entrada

Faça o upload dos **três** arquivos:

| Arquivo | Conteúdo |
|---------|----------|
| `precipitacaocajazeiras.csv` | P mensal — Cajazeiras (Eng. Ávidos) |
| `precipitacaosousa.csv` | P mensal — Sousa (São Gonçalo) |
| `vazoes_mensais_soma.csv` | Σ Q diária por mês (m³/s) — observada |

> **Alternativa Drive:** comente o bloco `upload` e descomente o bloco `drive.mount`.

In [ ]:
from google.colab import files as colab_files

# ── Opção A: upload direto ──────────────────────────────────────────────────
uploaded = colab_files.upload()
print('Carregados:', list(uploaded.keys()))

# ── Opção B: Google Drive (descomente se preferir) ──────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/pasta_dos_dados')

Saving vazoes_mensais_soma.csv to vazoes_mensais_soma (1).csv
Saving precipitacaocajazeiras.csv to precipitacaocajazeiras (1).csv
Saving precipitacaosousa.csv to precipitacaosousa (1).csv
Carregados: ['vazoes_mensais_soma (1).csv', 'precipitacaocajazeiras (1).csv', 'precipitacaosousa (1).csv']


## Importações de Bibliotecas

In [ ]:
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.optimize import minimize, differential_evolution

%matplotlib inline
print('Bibliotecas importadas.')

## Constantes e Configurações

Edite apenas esta célula para ajustar parâmetros, reservatórios ou janela de calibração.

| Bloco | Parâmetros principais |
|-------|-----------------------|
| **Simulação** | STR=1250 mm · PES=4.5 · CREC=0% · K=1 · Kp=0.75 |
| **Calibração** | Período 1985–2012 · Split 70/30 · Bounds semiárido cristalino |

In [ ]:
# ── Evaporação Piche — climatologia INMET São Gonçalo (mm/mês, Jan→Dez) ──────
E_PICHE_CLIMATOLOGIA = [
    185.5, 135.4, 122.5, 119.8, 139.8, 155.7,
    188.4, 230.2, 250.6, 252.8, 244.0, 224.7,
]

KP_MENSAL = [0.75] * 12   # coeficiente Piche → ETP (calibrável por mês)

MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
            'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

RESERVATORIOS = {
    'Engenheiro Ávidos': {'csv': 'precipitacaocajazeiras.csv', 'AD_km2': 942.11},
    'São Gonçalo':       {'csv': 'precipitacaosousa.csv',      'AD_km2': 306.10},
}

# ── Parâmetros SMAP validados — AESA (2016) ───────────────────────────────────
PARAMS_AESA = {'STR': 1250.0, 'PES': 4.5, 'CREC': 0.0, 'K': 1.0}

# ── Condições iniciais (estado estimado da bacia antes de jan/1963) ───────────
COND_INICIAIS = {'R_in': 0.0, 'RS_in': 4.0, 'Eb_in': 1.051}

# ── Janela temporal com dados fluviométricos confiáveis ───────────────────────
PERIODO_INICIO = '1985-01-01'
PERIODO_FIM    = '2012-12-31'
SPLIT_CAL_VAL  = 0.70          # 70% calibração / 30% validação

CSV_VAZOES = 'vazoes_mensais_soma.csv'

# ── Limites dos parâmetros para o semiárido cristalino ────────────────────────
BOUNDS = [
    (100.0, 2000.0),   # STR  [mm]  — capacidade do reservatório do solo
    (0.5,   7.0),      # PES  [–]   — expoente do escoamento superficial
    (0.0,  20.0),      # CREC [%]   — recarga; faixa ampla corrige subestimação de volume
    (0.2,   1.0),      # K    [–]   — recessão do escoamento de base
]

# ── Critérios de aceitação da calibração — Moriasi et al. (2007) ──────────────
NSE_MIN_OK     = 0.50    # NSE satisfatório (cal e val)
PBIAS_MAX_OK   = 25.0    # |PBIAS| satisfatório (%)
CAL_MAX_ROUNDS = 6       # rodadas de recalibração com peso de viés crescente

# ── Aquecimento do modelo (warm-up / spin-up) ─────────────────────────────────
# As condições iniciais (R_in, RS_in, Eb_in) são ESTADOS do sistema, não
# descritores físicos transferíveis entre bacias. Por isso NÃO são regionalizadas
# para a bacia não monitorada (Eng. Ávidos): em vez de "chutar" um Eb_in por área,
# simula-se um período inicial de aquecimento cujos resultados são descartados,
# eliminando a dependência do estado inicial arbitrário. É a prática-padrão em
# modelagem hidrológica (período de spin-up), aplicada de forma idêntica à
# calibração (São Gonçalo) e às simulações de ambas as bacias.
WARMUP_ANOS  = 2
WARMUP_MESES = WARMUP_ANOS * 12

print('Constantes definidas.')
print(f'  Reservatórios : {list(RESERVATORIOS.keys())}')
print(f'  PARAMS_AESA   : {PARAMS_AESA}')
print(f'  Período Cal   : {PERIODO_INICIO[:4]}–{PERIODO_FIM[:4]}')

## `load_precipitation` — Precipitação Mensal

Lê o CSV no formato largo (*wide*) e converte para série temporal via `pandas.melt()`.

```
wide → long:  Ano/mes | Jan | Fev …   →   DatetimeIndex | precipitacao_mm
```

In [ ]:
def load_precipitation(filepath: str) -> pd.Series:
    """Lê CSV largo de precipitação → pd.Series com DatetimeIndex mensal."""
    df = pd.read_csv(filepath)
    df = df.dropna(how='all')
    df = df.dropna(subset=['Ano/mes'])
    df['Ano/mes'] = df['Ano/mes'].astype(int)
    df[MESES_PT]  = df[MESES_PT].fillna(0.0)

    df_long = df.melt(id_vars='Ano/mes', value_vars=MESES_PT,
                      var_name='mes_nome', value_name='precipitacao_mm')

    mapa = {n: i + 1 for i, n in enumerate(MESES_PT)}
    df_long['mes_num'] = df_long['mes_nome'].map(mapa)
    df_long['data'] = pd.to_datetime(
        df_long['Ano/mes'].astype(str) + '-' +
        df_long['mes_num'].astype(str).str.zfill(2) + '-01')

    df_long = df_long.sort_values('data').set_index('data')
    serie = df_long['precipitacao_mm']

    # Reindexa para um calendário mensal contíguo (MS): anos/meses ausentes no
    # CSV recebem 0.0 mm. Garante frequência válida e evita o ValueError
    # "Inferred frequency None ... does not conform to passed frequency MS".
    idx_full = pd.date_range(serie.index.min(), serie.index.max(), freq='MS')
    serie = serie.reindex(idx_full, fill_value=0.0)
    serie.index.freq = 'MS'
    return serie

## `load_observed_flow` — Vazão Observada (Q_obs)

Lê `vazoes_mensais_soma.csv` (mesmo formato largo da precipitação) e aplica a
conversão crítica:

$$Q_{\text{média}} \; [\text{m}^3/\text{s}] = \frac{\sum Q_{\text{diária}}}{n_{\text{dias}}}$$

O `index.days_in_month` do pandas retorna o $n_{\text{dias}}$ correto para cada mês,
vetorizando a divisão e respeitando anos bissextos.
Valores ausentes no CSV são preservados como **NaN** — as métricas os ignoram automaticamente.

In [ ]:
def load_observed_flow(filepath: str) -> pd.Series:
    """
    Lê CSV de vazões (SOMA diária/mês) → pd.Series com DatetimeIndex mensal.
    Converte: Q_média = Σ Q_dia / n_dias (index.days_in_month).
    NaN onde dado ausente ou inválido.
    """
    df = pd.read_csv(filepath)
    df = df.dropna(how='all')
    col_ano = df.columns[0]
    df = df.dropna(subset=[col_ano])
    df[col_ano] = df[col_ano].astype(int)

    meses_ok = [m for m in MESES_PT if m in df.columns]
    df[meses_ok] = df[meses_ok].apply(pd.to_numeric, errors='coerce')

    df_long = df.melt(id_vars=col_ano, value_vars=meses_ok,
                      var_name='mes_nome', value_name='q_soma')

    mapa = {n: i + 1 for i, n in enumerate(MESES_PT)}
    df_long['mes_num'] = df_long['mes_nome'].map(mapa)
    df_long['data'] = pd.to_datetime(
        df_long[col_ano].astype(str) + '-' +
        df_long['mes_num'].astype(str).str.zfill(2) + '-01')

    df_long = df_long.sort_values('data').set_index('data')
    serie = df_long['q_soma']

    # Reindexa para calendário mensal contíguo (MS); meses ausentes ficam NaN
    # (sem vazão observada). Frequência válida evita o ValueError de MS.
    idx_full = pd.date_range(serie.index.min(), serie.index.max(), freq='MS')
    serie = serie.reindex(idx_full)
    serie.index.freq = 'MS'

    # Soma diária ÷ dias do mês → Média Mensal (vetorizado)
    serie = serie / serie.index.days_in_month
    return serie

## `build_etp_series` — ETP por Climatologia Piche × Kp

$$EP_{\text{mês}} = E_{\text{Piche, mês}} \times Kp_{\text{mês}}$$

Série estacionária: o mesmo perfil sazonal se repete em todos os anos.
Vetorizada via indexação numpy: `ep_array[index.month - 1]`.

In [ ]:
def build_etp_series(index, e_piche=None, kp_mensal=None):
    """Constrói série ETP replicando climatologia Piche × Kp (vetorizada)."""
    if e_piche  is None: e_piche  = E_PICHE_CLIMATOLOGIA
    if kp_mensal is None: kp_mensal = KP_MENSAL
    ep_clim  = [ep * kp for ep, kp in zip(e_piche, kp_mensal)]
    ep_array = np.array(ep_clim)
    return pd.Series(ep_array[index.month - 1], index=index, name='EP_mm')

## `smap_step` — Passo Temporal do SMAP

Executa **um mês** de simulação. Chamado sequencialmente por `run_smap`.

| Regime | Equações chave |
|--------|----------------|
| **Úmido** P ≥ EP | $ER = EP$ · $ES = P \cdot TU^{PES}$ · $REC = (CREC/100) \cdot R$ |
| **Seco** P < EP | $ER = \min(P+R,\; EP \cdot (2TU - TU^2))$ · $ES = 0$ |
| **Aquífero** | $Eb = K \cdot RS_{\text{new}}$ · $Q = ET_{\text{mm}} \times 10^{-3} \times AD / (n_{\text{d}} \times 86400)$ |

In [ ]:
def smap_step(P, EP, R, RS, params, AD_km2, n_dias):
    """Um passo mensal SMAP. Retorna dict com ES, ER, REC, Eb_mm, ET_mm, R_new, RS_new, Q_m3s, TU."""
    STR  = params['STR']
    PES  = params['PES']
    CREC = params['CREC']
    K    = params['K']

    TU = float(np.clip(R / STR, 0.0, 1.0))

    if P >= EP:                                   # ── Caso úmido ──
        ER  = EP
        ES  = P * (TU ** PES)
        REC = (CREC / 100.0) * R
        R_new = R + P - ER - ES - REC
        if R_new > STR: ES += (R_new - STR); R_new = STR
        if R_new < 0.0: ER += R_new;          R_new = 0.0
    else:                                         # ── Caso seco ──
        er_bud = EP * (2.0 * TU - TU ** 2)
        ER  = max(0.0, float(min(P + R, er_bud)))
        ES  = 0.0;  REC = 0.0
        R_new = R + P - ER
        if R_new < 0.0: ER += R_new; R_new = 0.0

    RS_new = RS + REC
    Eb_mm  = K * RS_new
    RS_new = RS_new - Eb_mm
    if RS_new < 0.0: Eb_mm += RS_new; RS_new = 0.0
    Eb_mm = max(0.0, Eb_mm)

    ET_mm = ES + Eb_mm
    Q_m3s = ET_mm * 1e-3 * AD_km2 * 1e6 / (n_dias * 86400)

    return {'ER': ER, 'ES': ES, 'REC': REC, 'Eb_mm': Eb_mm,
            'ET_mm': ET_mm, 'R_new': R_new, 'RS_new': RS_new,
            'Q_m3s': Q_m3s, 'TU': TU}

## `run_smap` — Simulação Completa da Série Temporal

Loop mensal sequencial: o estado $(R_t, RS_t)$ de cada mês alimenta o mês seguinte.
`Eb_in` é convertido de m³/s para mm e somado ao `RS_in` para preservar a memória
hidrológica do rio antes do início da série.

In [ ]:
def run_smap(P_series, EP_series, params, cond_iniciais, AD_km2):
    """Loop temporal SMAP → pd.DataFrame com P_mm, EP_mm, R_mm, RS_mm, TU, ER_mm, ES_mm, REC_mm, Eb_mm, ET_mm, Q_m3s."""
    R = float(cond_iniciais['R_in'])

    # Converte Eb_in (m³/s) → mm e adiciona ao RS_in: preserva memória pré-simulação
    d0     = P_series.index[0]
    nd0    = calendar.monthrange(d0.year, d0.month)[1]
    eb_mm0 = cond_iniciais['Eb_in'] * nd0 * 86400 / (AD_km2 * 1e3)
    RS     = float(cond_iniciais['RS_in']) + eb_mm0

    registros = []
    for data in P_series.index:
        P_mes  = float(P_series[data])
        EP_mes = float(EP_series[data])
        n_dias = calendar.monthrange(data.year, data.month)[1]
        res    = smap_step(P_mes, EP_mes, R, RS, params, AD_km2, n_dias)
        registros.append({'data': data, 'P_mm': P_mes, 'EP_mm': EP_mes,
                          'R_mm': R, 'RS_mm': RS, 'TU': res['TU'],
                          'ER_mm': res['ER'], 'ES_mm': res['ES'],
                          'REC_mm': res['REC'], 'Eb_mm': res['Eb_mm'],
                          'ET_mm': res['ET_mm'], 'Q_m3s': res['Q_m3s']})
        R  = res['R_new']
        RS = res['RS_new']

    df = pd.DataFrame(registros).set_index('data')
    # infer_freq retorna 'MS' para índices mensais contíguos e None caso haja
    # lacunas — evita o ValueError ao atribuir a frequência diretamente.
    df.index = pd.DatetimeIndex(df.index)
    df.index.freq = pd.infer_freq(df.index)
    return df

## Métricas de Desempenho Hidrológico

| Métrica | Fórmula | Ideal |
|---------|---------|-------|
| **NSE** | $1 - \Sigma(Q_o-Q_s)^2 / \Sigma(Q_o-\bar Q_o)^2$ | 1.0 |
| **PBIAS** | $100 \cdot \Sigma(Q_o-Q_s) / \Sigma Q_o$ | 0% |

Classificação (Moriasi et al., 2007): NSE > 0.75 = Muito Bom · > 0.65 = Bom · > 0.50 = Satisfatório
Pares NaN/Inf são descartados automaticamente.

In [ ]:
def nse(Q_obs, Q_sim):
    """Nash-Sutcliffe Efficiency. Ignora pares NaN/Inf."""
    Q_obs = np.asarray(Q_obs, dtype=float)
    Q_sim = np.asarray(Q_sim, dtype=float)
    ok = np.isfinite(Q_obs) & np.isfinite(Q_sim)
    if ok.sum() < 2: return np.nan
    q_o = Q_obs[ok]; q_s = Q_sim[ok]
    sst = np.sum((q_o - q_o.mean()) ** 2)
    if sst < 1e-12: return np.nan
    return float(1.0 - np.sum((q_o - q_s) ** 2) / sst)


def pbias(Q_obs, Q_sim):
    """PBIAS (%). > 0 = subestimação. Ignora pares NaN/Inf."""
    Q_obs = np.asarray(Q_obs, dtype=float)
    Q_sim = np.asarray(Q_sim, dtype=float)
    ok = np.isfinite(Q_obs) & np.isfinite(Q_sim)
    q_o = Q_obs[ok]; q_s = Q_sim[ok]
    if q_o.sum() < 1e-12: return np.nan
    return float(100.0 * (q_o - q_s).sum() / q_o.sum())


def classify_nse(v):
    """Classificação NSE — Moriasi et al. (2007)."""
    if not np.isfinite(v): return 'N/D'
    if v > 0.75: return 'Muito Bom'
    if v > 0.65: return 'Bom'
    if v > 0.50: return 'Satisfatório'
    return 'Insatisfatório'


def classify_pbias(v):
    """Classificação |PBIAS| — Moriasi et al. (2007)."""
    if not np.isfinite(v): return 'N/D'
    a = abs(v)
    if a <= 10: return 'Muito Bom'
    if a <= 15: return 'Bom'
    if a <= 25: return 'Satisfatório'
    return 'Insatisfatório'

def kge(Q_obs, Q_sim):
    """Kling-Gupta Efficiency (Gupta et al., 2009). Decompõe em r, alpha (variabilidade)
    e beta (viés = média_sim/média_obs). KGE=1 perfeito. Ignora pares NaN/Inf."""
    Q_obs = np.asarray(Q_obs, dtype=float)
    Q_sim = np.asarray(Q_sim, dtype=float)
    ok = np.isfinite(Q_obs) & np.isfinite(Q_sim)
    if ok.sum() < 2: return np.nan
    q_o = Q_obs[ok]; q_s = Q_sim[ok]
    if q_o.std() < 1e-12 or q_s.std() < 1e-12 or q_o.mean() <= 0: return np.nan
    r     = np.corrcoef(q_o, q_s)[0, 1]
    alpha = q_s.std()  / q_o.std()
    beta  = q_s.mean() / q_o.mean()
    return float(1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2))

## Calibração — `objective_function` e `calibrate`

**Objetivo:** minimizar a **distância KGE** (Gupta et al., 2009) — que decompõe o erro em
correlação ($r$), variabilidade ($\alpha$) e **viés** ($\beta = \overline{Q_{sim}}/\overline{Q_{obs}}$).
Otimizar $\beta \to 1$ ataca diretamente o **PBIAS**, ao contrário do NSE puro.
**Otimizador:** **Evolução Diferencial** (busca global, escapa de mínimos locais).
**Recalibração automática:** rodadas com **peso de viés crescente** até que NSE e
$|PBIAS|$ fiquem *satisfatórios* (Moriasi, 2007) em calibração **e** validação.
**Split cronológico (Klemeš, 1986):** 70 % calibração · 30 % validação.

In [ ]:
def objective_function(params_vec, P_series, EP_series, Q_obs, AD_km2, cal_mask, w_bias=1.0):
    """
    Distância KGE generalizada na janela de calibração (a MINIMIZAR).
    O peso `w_bias` amplifica o termo de viés (beta), forçando o otimizador a
    reduzir o PBIAS. Penalidade 1e6 em caso de falha numérica.
    """
    params = {'STR': float(params_vec[0]), 'PES': float(params_vec[1]),
              'CREC': float(params_vec[2]), 'K': float(params_vec[3])}
    try:
        df_sim = run_smap(P_series, EP_series, params, COND_INICIAIS, AD_km2)
        o = Q_obs.values[cal_mask]
        s = df_sim['Q_m3s'].values[cal_mask]
        ok = np.isfinite(o) & np.isfinite(s)
        if ok.sum() < 2: return 1e6
        o = o[ok]; s = s[ok]
        mo, ms = o.mean(), s.mean()
        so, ss = o.std(),  s.std()
        if mo <= 0 or so < 1e-12 or ss < 1e-12: return 1e6
        r = np.corrcoef(o, s)[0, 1]
        if not np.isfinite(r): return 1e6
        alpha = ss / so                       # razão de variabilidade
        beta  = ms / mo                       # razão de viés (beta=1 -> PBIAS=0)
        ed = np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (w_bias * (beta - 1.0)) ** 2)
        return float(ed) if np.isfinite(ed) else 1e6
    except Exception:
        return 1e6


def calibrate(P_series, EP_series, Q_obs, AD_km2, split=SPLIT_CAL_VAL, warmup=0,
              nse_min=NSE_MIN_OK, pbias_max=PBIAS_MAX_OK,
              max_rounds=CAL_MAX_ROUNDS, seed=42, verbose=True):
    """
    Calibra {STR, PES, CREC, K} por Evolução Diferencial (busca global) sobre a
    distância KGE.

    Rigor metodológico:
    - A SELEÇÃO dos parâmetros usa SOMENTE a janela de calibração; a validação
      permanece um teste independente (split-sample, Klemeš 1986) — sem vazamento.
    - As rodadas com peso de viés crescente só prosseguem enquanto o |PBIAS| de
      calibração estiver acima do limiar; satisfeita a calibração, encerra.

    Parâmetros:
    - `warmup` (meses): aquecimento (spin-up) excluído das métricas e do split.
    - `split`: fração de calibração. Use `split=1.0` para CALIBRAÇÃO OPERACIONAL
      no período completo (sem validação) — conjunto transferido p/ bacia vizinha.

    Retorna dict com params_cal, split_date, máscaras, Q_sim/Q_obs (pós-aquecimento),
    nse/pbias/kge de cal e val, n_cal/n_val, converged, rounds_run, satisfied.
    """
    n      = len(P_series)
    idx    = P_series.index
    n_eval = n - warmup
    full   = (split >= 1.0)

    if full:
        n_cal = n_eval
        sd    = None
        cal_mask = np.array(idx >= idx[warmup], dtype=bool)
        val_mask = np.zeros(n, dtype=bool)
    else:
        n_cal = int(split * n_eval)
        sd    = idx[warmup + n_cal]
        cal_mask = np.array((idx >= idx[warmup]) & (idx < sd), dtype=bool)
        val_mask = np.array(idx >= sd, dtype=bool)
    n_val = n_eval - n_cal

    if verbose:
        if warmup > 0:
            print(f'  Aquecimento: {idx[0].date()} → {idx[warmup-1].date()} ({warmup} meses, descartado)')
        if full:
            print(f'  Calibração (período completo): {idx[warmup].date()} → {idx[-1].date()} ({n_cal} meses)')
        else:
            print(f'  Cal: {idx[warmup].date()} → {idx[warmup + n_cal - 1].date()} ({n_cal} meses)')
            print(f'  Val: {sd.date()} → {idx[-1].date()} ({n_val} meses)')

    def evaluate(pc, converged):
        df_sim = run_smap(P_series, EP_series, pc, COND_INICIAIS, AD_km2)
        Q_sim  = df_sim['Q_m3s']
        Qoc = Q_obs.values[cal_mask];  Qsc = Q_sim.values[cal_mask]
        Qov = Q_obs.values[val_mask];  Qsv = Q_sim.values[val_mask]
        return {'params_cal': pc, 'split_date': sd,
                'cal_mask': cal_mask, 'val_mask': val_mask,
                'Q_sim': Q_sim.iloc[warmup:], 'Q_obs': Q_obs.iloc[warmup:],
                'nse_cal':   nse(Qoc, Qsc), 'pbias_cal': pbias(Qoc, Qsc),
                'nse_val':   nse(Qov, Qsv), 'pbias_val': pbias(Qov, Qsv),
                'kge_cal':   kge(Qoc, Qsc), 'kge_val':   kge(Qov, Qsv),
                'n_cal': n_cal, 'n_val': n_val,
                'converged': converged}

    def cal_ok(res):
        return (np.isfinite(res['nse_cal'])   and res['nse_cal']   >= nse_min and
                np.isfinite(res['pbias_cal']) and abs(res['pbias_cal']) <= pbias_max)

    def is_satisfied(res):
        if full: return cal_ok(res)
        val_ok = (np.isfinite(res['nse_val'])   and res['nse_val']   >= nse_min and
                  np.isfinite(res['pbias_val']) and abs(res['pbias_val']) <= pbias_max)
        return cal_ok(res) and val_ok

    def deficit_cal(res):            # seleção SÓ pela calibração (val independente)
        nse_c = res['nse_cal']      if np.isfinite(res['nse_cal'])   else -1.0
        pb_c  = abs(res['pbias_cal']) if np.isfinite(res['pbias_cal']) else 1e3
        return max(0.0, nse_min - nse_c) + max(0.0, pb_c - pbias_max) / 100.0

    pesos_vies = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0]
    best, best_def = None, np.inf

    for rnd in range(max_rounds):
        w = pesos_vies[min(rnd, len(pesos_vies) - 1)]
        opt = differential_evolution(
            objective_function, bounds=BOUNDS,
            args=(P_series, EP_series, Q_obs, AD_km2, cal_mask, w),
            seed=seed + rnd, maxiter=100, popsize=15, tol=1e-7,
            mutation=(0.5, 1.0), recombination=0.7, polish=True)

        pc = {'STR': round(float(opt.x[0]), 2), 'PES':  round(float(opt.x[1]), 4),
              'CREC': round(float(opt.x[2]), 4), 'K':   round(float(opt.x[3]), 4)}
        res = evaluate(pc, bool(opt.success))
        d   = deficit_cal(res)

        if verbose:
            extra = '' if full else f'  |  NSE val = {res["nse_val"]:.3f}  PBIAS val = {res["pbias_val"]:+.1f}%'
            print(f'  Rodada {rnd+1}/{max_rounds} (peso viés={w:>4.1f}): '
                  f'NSE cal = {res["nse_cal"]:.3f}  PBIAS cal = {res["pbias_cal"]:+.1f}%' + extra)

        if d < best_def:
            best, best_def = res, d
        if cal_ok(res):             # objetivo de calibração atingido → encerra
            break

    best['rounds_run'] = rnd + 1
    best['satisfied']  = is_satisfied(best)
    if verbose and not best['satisfied']:
        if full:
            print('  ⚠ Calibração não atingiu os limiares — retornando o melhor conjunto.')
        else:
            print('  ⚠ Validação abaixo do limiar (transferibilidade temporal) — ver calibração operacional.')
    return best

## `plot_simulation` — Hietograma + Hidrograma + Umidade do Solo

Dois painéis: (1) barras P invertidas (eixo Y direito) + linha Q (esquerdo),
(2) série de umidade R(t) com referência em STR.

In [ ]:
def plot_simulation(df, nome_reservatorio, params):
    """Gera figura 2-painéis: hietograma+hidrograma e umidade do solo."""
    sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)

    str_v  = params['STR']
    pes_v  = params['PES']
    crec_v = params['CREC']
    k_v    = params['K']

    fig = plt.figure(figsize=(16, 9))
    gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.40, height_ratios=[2, 1.2])
    titulo = (f'Simulação SMAP Mensal — Reservatório {nome_reservatorio}' + chr(10) +
              f'STR={str_v} mm  |  PES={pes_v}  |  CREC={crec_v}%  |  K={k_v}')
    fig.suptitle(titulo, fontsize=13, fontweight='bold', y=0.99)

    ax1      = fig.add_subplot(gs[0])
    ax1_prec = ax1.twinx()

    ax1.plot(df.index, df['Q_m3s'], color='steelblue', linewidth=1.3, zorder=3,
             label='Q$_{sim}$ SMAP (m³/s)')
    ax1.fill_between(df.index, df['Q_m3s'], alpha=0.20, color='steelblue', zorder=2)
    ax1.set_ylabel('Vazão (m³/s)', color='steelblue', fontsize=10)
    ax1.tick_params(axis='y', labelcolor='steelblue')
    ax1.set_ylim(bottom=0)

    ax1_prec.bar(df.index, df['P_mm'], width=20, color='navy',
                 alpha=0.55, zorder=1, label='Precipitação P (mm)')
    ax1_prec.set_ylabel('Precipitação (mm)', color='navy', fontsize=10)
    ax1_prec.tick_params(axis='y', labelcolor='navy')
    ax1_prec.invert_yaxis()

    lq, llq = ax1.get_legend_handles_labels()
    lp, llp = ax1_prec.get_legend_handles_labels()
    ax1.legend(lq + lp, llq + llp, loc='upper right', fontsize=9)
    ax1.set_title('Hietograma e Hidrograma Simulado', fontsize=11, pad=4)

    q_med = df['Q_m3s'].mean()
    q_max = df['Q_m3s'].max()
    secos = (df['Q_m3s'] < 0.001).sum()
    ax1.text(0.01, 0.97,
             f'Q̅={q_med:.2f} m³/s  |  Q_máx={q_max:.1f} m³/s  |  Meses secos: {secos}/{len(df)}',
             transform=ax1.transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.fill_between(df.index, df['R_mm'], 0, alpha=0.55, color='saddlebrown',
                     label='R — Umidade do Solo (mm)')
    ax2.axhline(str_v, color='crimson', linestyle='--', linewidth=1.2,
                label=f'STR = {str_v} mm (capacidade máxima)')
    ax2.set_ylabel('R (mm)', fontsize=10)
    ax2.set_ylim(0, str_v * 1.15)
    ax2.set_xlabel('Data', fontsize=10)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.set_title('Umidade do Solo (Reservatório Linear)', fontsize=11, pad=4)

    plt.tight_layout()
    nome_arq = ('smap_' + nome_reservatorio.lower()
                .replace(' ', '_').replace('.', '')
                .replace('ã', 'a').replace('é', 'e').replace('ó', 'o') + '.png')
    plt.savefig(nome_arq, dpi=150, bbox_inches='tight')
    print(f'  → Figura salva: {nome_arq}')
    plt.show()

## `save_flow_csv` — Exportação de Vazões Afluentes

In [ ]:
def save_flow_csv(df_smap, nome_reservatorio):
    """Exporta Q_m3s para CSV no formato largo (mesmo layout da entrada)."""
    df_aux = pd.DataFrame({
        'Ano/mes':  df_smap.index.year,
        'mes_nome': df_smap.index.month.map({i+1: n for i, n in enumerate(MESES_PT)}),
        'Q_m3s':    df_smap['Q_m3s'].values,
    })
    df_wide = df_aux.pivot_table(index='Ano/mes', columns='mes_nome',
                                 values='Q_m3s', aggfunc='first')
    df_wide = df_wide[MESES_PT].round(4)
    df_wide.columns.name = None
    nome_arq = ('vazoes_afluentes_' + nome_reservatorio.lower()
                .replace(' ', '_').replace('.', '')
                .replace('ã', 'a').replace('é', 'e').replace('ó', 'o') + '.csv')
    df_wide.to_csv(nome_arq, sep=',', decimal='.')
    return nome_arq

## `plot_calibration` — Série Temporal Q_obs × Q_sim

• Q_obs: linha cinza (NaN → lacunas naturais, sem conexão)
• Q_sim: linha azul
• Linha vertical vermelha pontilhada: corte exato cal/val
• Anotações de NSE e PBIAS por período

In [ ]:
def plot_calibration(res, nome_reservatorio):
    """Série temporal Q_obs × Q_sim com linha vertical cal/val."""
    sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)

    Q_obs  = res['Q_obs'];   Q_sim = res['Q_sim']
    split  = res['split_date']
    p      = res['params_cal']
    str_v  = p['STR'];  pes_v = p['PES'];  crec_v = p['CREC'];  k_v = p['K']
    nse_c  = res['nse_cal'];  pb_c = res['pbias_cal']
    nse_v  = res['nse_val'];  pb_v = res['pbias_val']
    n_c    = res['n_cal'];    n_v  = res['n_val']

    fig, ax = plt.subplots(figsize=(16, 6))

    # NaN em Q_obs gera lacunas automáticas (matplotlib não conecta sobre NaN)
    ax.plot(Q_obs.index, Q_obs.values, color='dimgray', linewidth=1.0, alpha=0.85,
            zorder=3, label='Q$_{obs}$ (observada)')
    ax.plot(Q_sim.index, Q_sim.values, color='steelblue', linewidth=1.4,
            zorder=4, label='Q$_{sim}$ (SMAP calibrado)')
    ax.axvline(split, color='crimson', linestyle='--', linewidth=1.8, zorder=5,
               label=f'Corte Cal/Val: {split.strftime("%b/%Y")}')
    ax.axvspan(Q_obs.index[0], split, alpha=0.04, color='steelblue')
    ax.axvspan(split, Q_obs.index[-1], alpha=0.04, color='darkorange')

    nl = chr(10)
    ax.text(0.01, 0.97,
            f'CALIBRAÇÃO ({n_c} meses){nl}'
            f'NSE  = {nse_c:.3f}  [{classify_nse(nse_c)}]{nl}'
            f'PBIAS = {pb_c:.1f}%  [{classify_pbias(pb_c)}]',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round', facecolor='#dce8f7', alpha=0.90))
    ax.text(0.76, 0.97,
            f'VALIDAÇÃO ({n_v} meses){nl}'
            f'NSE  = {nse_v:.3f}  [{classify_nse(nse_v)}]{nl}'
            f'PBIAS = {pb_v:.1f}%  [{classify_pbias(pb_v)}]',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round', facecolor='#fef3c7', alpha=0.90))

    ini = PERIODO_INICIO[:4];  fim = PERIODO_FIM[:4]
    titulo = (f'Calibração/Validação SMAP — Reservatório {nome_reservatorio} | {ini}–{fim}' + chr(10) +
              f'STR={str_v} mm  |  PES={pes_v}  |  CREC={crec_v}%  |  K={k_v}')
    fig.suptitle(titulo, fontsize=12, fontweight='bold', y=1.01)
    ax.set_ylabel('Vazão (m³/s)', fontsize=10)
    ax.set_xlabel('Data', fontsize=10)
    ax.set_ylim(bottom=0)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    nome_arq = ('calibracao_' + nome_reservatorio.lower()
                .replace(' ', '_').replace('.', '')
                .replace('ã', 'a').replace('é', 'e').replace('ó', 'o') + '.png')
    plt.savefig(nome_arq, dpi=150, bbox_inches='tight')
    print(f'  → Figura salva: {nome_arq}')
    plt.show()

---
## Execução

**São Gonçalo** é a única bacia com vazões observadas → calibração do modelo.
**Engenheiro Ávidos** é bacia vizinha sem dados → recebe os parâmetros
calibrados de São Gonçalo (regionalização por proximidade).

In [ ]:
# Vazões observadas mensais — disponíveis SOMENTE para São Gonçalo (posto
# fluviométrico). Engenheiro Ávidos é tratado como bacia NÃO monitorada.
Q_obs_full = load_observed_flow(CSV_VAZOES)
n_tot = len(Q_obs_full)
n_nan = Q_obs_full.isna().sum()
print(f'Q_obs (São Gonçalo): {Q_obs_full.index[0].date()} → {Q_obs_full.index[-1].date()} ({n_tot} meses)')
print(f'Faltantes: {n_nan}/{n_tot} ({100*n_nan/n_tot:.1f}%)')

### São Gonçalo — Calibração (teste split-sample + calibração operacional)

Bacia **monitorada** (única com vazões observadas). Duas etapas:

1. **Teste split-sample** (Klemeš, 1986): calibra em 70 % (1985–2004) e **valida**
   nos 30 % finais (2004–2012) — mede a transferibilidade temporal. A validação é
   *independente* (não entra na escolha dos parâmetros).
2. **Calibração operacional**: recalibra no **período completo** (1985–2012),
   aproveitando toda a informação observada. É este conjunto (`PARAMS_SG_CAL`)
   que é transferido para Engenheiro Ávidos.

Ambas usam aquecimento (spin-up) anterior a 1985, descartado das métricas.

In [ ]:
nome_sg   = 'São Gonçalo'
config_sg = RESERVATORIOS[nome_sg]
AD_sg     = config_sg['AD_km2']

# Séries da bacia monitorada
P_sg  = load_precipitation(config_sg['csv'])
EP_sg = build_etp_series(P_sg.index)

# Recorte: WARMUP_ANOS anos de aquecimento ANTES de PERIODO_INICIO + período observado
ini_warm  = (pd.Timestamp(PERIODO_INICIO) - pd.DateOffset(years=WARMUP_ANOS)).strftime('%Y-%m-%d')
P_sg_fil  = P_sg[ini_warm:PERIODO_FIM]
EP_sg_fil = EP_sg[ini_warm:PERIODO_FIM]
Q_obs_sg  = Q_obs_full.reindex(P_sg_fil.index)   # NaN no aquecimento e onde ausente

n_valid_sg = int(Q_obs_sg.notna().sum())
ini = PERIODO_INICIO[:4];  fim = PERIODO_FIM[:4]
print(f'Reservatório     : {nome_sg}  |  AD = {AD_sg} km²')
print(f'Aquecimento      : {WARMUP_ANOS} anos antes de {ini} (descartado das métricas)')
print(f'Período avaliado : {ini}–{fim}  |  Q_obs válidas: {n_valid_sg}')

# Referência AESA (antes da calibração)
df_ref_sg = run_smap(P_sg_fil, EP_sg_fil, PARAMS_AESA, COND_INICIAIS, AD_sg)
nse_a = nse(Q_obs_sg.values, df_ref_sg['Q_m3s'].values)
pb_a  = pbias(Q_obs_sg.values, df_ref_sg['Q_m3s'].values)
print(f'\nAESA : STR={PARAMS_AESA["STR"]} mm  PES={PARAMS_AESA["PES"]}  CREC={PARAMS_AESA["CREC"]}%  K={PARAMS_AESA["K"]}')
print(f'  NSE   = {nse_a:.3f}  [{classify_nse(nse_a)}]  |  PBIAS = {pb_a:.1f}%  [{classify_pbias(pb_a)}]')

# ─────────────────────────────────────────────────────────────────────────────
# (1) TESTE SPLIT-SAMPLE (Klemeš, 1986): calibra em 70% e valida nos 30% finais.
#     Mede a transferibilidade temporal. A validação NÃO entra na escolha dos
#     parâmetros (teste independente).
# ─────────────────────────────────────────────────────────────────────────────
print('\n══ Teste split-sample (calibração 70% · validação 30%) ══')
res_sg_test = calibrate(P_sg_fil, EP_sg_fil, Q_obs_sg, AD_sg, warmup=WARMUP_MESES)
print(f'\nCalibração ({res_sg_test["n_cal"]} meses):')
print(f'  NSE = {res_sg_test["nse_cal"]:.3f}  [{classify_nse(res_sg_test["nse_cal"])}]  |  PBIAS = {res_sg_test["pbias_cal"]:.1f}%  [{classify_pbias(res_sg_test["pbias_cal"])}]')
print(f'Validação ({res_sg_test["n_val"]} meses):')
print(f'  NSE = {res_sg_test["nse_val"]:.3f}  [{classify_nse(res_sg_test["nse_val"])}]  |  PBIAS = {res_sg_test["pbias_val"]:.1f}%  [{classify_pbias(res_sg_test["pbias_val"])}]')
print(f'  KGE = {res_sg_test["kge_cal"]:.3f} (cal)  |  {res_sg_test["kge_val"]:.3f} (val)')
plot_calibration(res_sg_test, nome_sg)

# ─────────────────────────────────────────────────────────────────────────────
# (2) CALIBRAÇÃO OPERACIONAL no período COMPLETO (split=1.0): usa toda a série
#     observada 1985–2012. É este conjunto que será TRANSFERIDO para Eng. Ávidos.
# ─────────────────────────────────────────────────────────────────────────────
print('\n══ Calibração operacional (período completo → parâmetros transferidos) ══')
res_sg = calibrate(P_sg_fil, EP_sg_fil, Q_obs_sg, AD_sg, warmup=WARMUP_MESES, split=1.0)
PARAMS_SG_CAL = res_sg['params_cal']
conv = 'sim' if res_sg['converged'] else 'não'
print(f'\nParâmetros calibrados (convergência: {conv}):')
print(f'  STR={PARAMS_SG_CAL["STR"]} mm  PES={PARAMS_SG_CAL["PES"]}  CREC={PARAMS_SG_CAL["CREC"]}%  K={PARAMS_SG_CAL["K"]}')
print(f'\nDesempenho no período completo ({res_sg["n_cal"]} meses):')
print(f'  NSE = {res_sg["nse_cal"]:.3f}  [{classify_nse(res_sg["nse_cal"])}]  |  PBIAS = {res_sg["pbias_cal"]:.1f}%  [{classify_pbias(res_sg["pbias_cal"])}]  |  KGE = {res_sg["kge_cal"]:.3f}')

### Diagnóstico da série observada — São Gonçalo

Antes de qualquer reajuste de limites, examina-se a **qualidade e a
estacionariedade** da série observada nas janelas de calibração (1985–2004) e
validação (2004–2012): falhas, meses secos, estatísticas e a **relação
chuva–vazão** (coeficiente de escoamento *C* = Σlâmina/ΣP). Uma variação grande
de *C* entre as janelas indica **não-estacionariedade** — causa provável do NSE
de validação baixo e base para a discussão na tese.

In [ ]:
# ── Diagnóstico da série observada (São Gonçalo) ──────────────────────────────
# Fundamenta a queda do NSE de validação e a posição dos parâmetros nos limites.
split_dt = res_sg_test['split_date']                 # corte cal/val (do teste)

idx_av   = pd.date_range(PERIODO_INICIO, PERIODO_FIM, freq='MS')
Q_obs_av = Q_obs_full.reindex(idx_av)                # vazão observada (m³/s)
P_av     = P_sg.reindex(idx_av)                      # precipitação (mm)
# Q (m³/s) → lâmina escoada mensal (mm) na área da bacia
Qd_mm    = Q_obs_av * idx_av.days_in_month * 86400.0 / (AD_sg * 1e3)

def resumo(mask, nome):
    q  = Q_obs_av[mask];  qd = Qd_mm[mask];  p = P_av[mask]
    n      = int(mask.sum())
    n_obs  = int(q.notna().sum())
    falt   = 100.0 * (n - n_obs) / n
    secos  = 100.0 * (q < 0.01).sum() / max(n_obs, 1)
    qm     = q.mean();  cv = q.std() / qm if qm > 0 else float('nan')
    ok     = qd.notna() & p.notna()
    C      = qd[ok].sum() / p[ok].sum() if p[ok].sum() > 0 else float('nan')
    print(f'{nome:11}: {n_obs:>3} meses obs | faltantes {falt:4.1f}% | meses secos {secos:4.1f}%')
    print(f'{"":11}  Q méd {qm:6.3f} m³/s | CV {cv:4.2f} | P anual {p.mean()*12:5.0f} mm | C(Q/P) {C:5.3f}')
    return dict(C=C, qm=qm)

mask_cal = idx_av <  split_dt
mask_val = idx_av >= split_dt

print('═' * 68)
print(f'CALIBRAÇÃO  {idx_av[0].date()} → {(split_dt - pd.DateOffset(months=1)).date()}')
rc = resumo(mask_cal, 'CALIBRAÇÃO')
print(f'\nVALIDAÇÃO   {split_dt.date()} → {idx_av[-1].date()}')
rv = resumo(mask_val, 'VALIDAÇÃO')
print('═' * 68)

dC = 100.0 * (rv['C'] - rc['C']) / rc['C'] if rc['C'] else float('nan')
dQ = 100.0 * (rv['qm'] - rc['qm']) / rc['qm'] if rc['qm'] else float('nan')
print(f'Δ coef. de escoamento C (val vs cal): {dC:+.1f}%   |   Δ Q médio: {dQ:+.1f}%')
print('→ |ΔC| elevado = mudança na relação chuva–vazão (não-estacionariedade):')
print('  um único conjunto de parâmetros não transfere bem no tempo, o que')
print('  explica o NSE de validação baixo apesar do PBIAS controlado.')

# ── Figura: regime temporal + relação chuva–vazão por janela ──────────────────
fig, (axA, axB) = plt.subplots(1, 2, figsize=(16, 5),
                               gridspec_kw={'width_ratios': [2, 1]})
axA.plot(Q_obs_av.index, Q_obs_av.values, color='dimgray', lw=0.9, label='Q$_{obs}$ mensal')
axA.plot(Q_obs_av.index, Q_obs_av.rolling(12, min_periods=6).mean().values,
         color='crimson', lw=2.0, label='média móvel 12 meses')
axA.axvline(split_dt, color='navy', ls='--', lw=1.6, label='corte cal/val')
axA.axvspan(idx_av[0], split_dt, color='steelblue', alpha=0.05)
axA.axvspan(split_dt, idx_av[-1], color='darkorange', alpha=0.05)
axA.set_title('Vazão observada — São Gonçalo (regime cal × val)')
axA.set_ylabel('Q (m³/s)'); axA.set_xlabel('Data'); axA.set_ylim(bottom=0)
axA.legend(fontsize=8, loc='upper right')

axB.scatter(P_av[mask_cal], Qd_mm[mask_cal], s=14, color='steelblue', alpha=0.6,
            label=f'Cal  (C={rc["C"]:.3f})')
axB.scatter(P_av[mask_val], Qd_mm[mask_val], s=14, color='darkorange', alpha=0.6,
            label=f'Val  (C={rv["C"]:.3f})')
axB.set_title('Relação chuva–vazão')
axB.set_xlabel('P (mm/mês)'); axB.set_ylabel('Lâmina escoada (mm/mês)')
axB.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('diagnostico_sao_goncalo.png', dpi=150, bbox_inches='tight')
print('\n  → Figura salva: diagnostico_sao_goncalo.png')
plt.show()

### São Gonçalo — Simulação com parâmetros calibrados (1963–2019)

In [ ]:
# Simulação completa de São Gonçalo com os parâmetros calibrados
P_anual_sg  = P_sg.resample('YE').sum().mean()
EP_anual_sg = EP_sg.resample('YE').sum().mean()
print(f'Reservatório : {nome_sg}  |  AD = {AD_sg} km²')
print(f'Período      : {P_sg.index[0].date()} → {P_sg.index[-1].date()}  ({WARMUP_ANOS} anos de aquecimento descartados)')
print(f'P  anual     : {P_anual_sg:.1f} mm  |  EP anual: {EP_anual_sg:.1f} mm')
print(f'Parâmetros   : {PARAMS_SG_CAL}')
print('-' * 45)

# Simula 1963–2019 e DESCARTA o aquecimento → série independente da condição inicial
df_full_sg = run_smap(P_sg, EP_sg, PARAMS_SG_CAL, COND_INICIAIS, AD_sg)
df_smap_sg = df_full_sg.iloc[WARMUP_MESES:]

Q_med_sg = df_smap_sg['Q_m3s'].mean()
Q_max_sg = df_smap_sg['Q_m3s'].max()
n_sec_sg = (df_smap_sg['Q_m3s'] < 0.001).sum()
print(f'Q_sim média  : {Q_med_sg:.4f} m³/s')
print(f'Q_sim máxima : {Q_max_sg:.4f} m³/s')
print(f'Meses secos  : {n_sec_sg}/{len(df_smap_sg)} ({100*n_sec_sg/len(df_smap_sg):.1f}%)')

# captura o nome do CSV exportado → consumido pela célula de download final
csv_sg = save_flow_csv(df_smap_sg, nome_sg)
print(f'CSV salvo    : {csv_sg}')

plot_simulation(df_smap_sg, nome_sg, PARAMS_SG_CAL)

### Engenheiro Ávidos — Simulação por transferência de parâmetros (1963–2019)

Engenheiro Ávidos **não possui vazões observadas**. Por ser bacia vizinha e
hidrologicamente semelhante a São Gonçalo, aplica-se **regionalização por
proximidade**: transferem-se os **parâmetros** calibrados em São Gonçalo
(`PARAMS_SG_CAL`).

**As condições iniciais NÃO são transferidas** — elas são *estados* do sistema,
não descritores físicos da bacia. Em vez de arbitrar um `Eb_in` por área, usa-se
um estado inicial neutro comum e descarta-se um período de **aquecimento**
(spin-up, `WARMUP_ANOS`), removendo a dependência da condição inicial.
Permanecem próprias da bacia a **precipitação** e a **área de drenagem** (`AD`).

In [ ]:
nome_ea   = 'Engenheiro Ávidos'
config_ea = RESERVATORIOS[nome_ea]
AD_ea     = config_ea['AD_km2']

# Séries próprias da bacia (precipitação de Cajazeiras; ETP pela climatologia)
P_ea  = load_precipitation(config_ea['csv'])
EP_ea = build_etp_series(P_ea.index)

P_anual_ea  = P_ea.resample('YE').sum().mean()
EP_anual_ea = EP_ea.resample('YE').sum().mean()
print(f'Reservatório : {nome_ea}  |  AD = {AD_ea} km²')
print(f'Período      : {P_ea.index[0].date()} → {P_ea.index[-1].date()}  ({WARMUP_ANOS} anos de aquecimento descartados)')
print(f'P  anual     : {P_anual_ea:.1f} mm  |  EP anual: {EP_anual_ea:.1f} mm')
print(f'Parâmetros   : transferidos de São Gonçalo → {PARAMS_SG_CAL}')
print('-' * 45)

# Vazões geradas com os parâmetros calibrados em São Gonçalo (regionalização).
# Condição inicial NÃO transferida: estado neutro comum + descarte do aquecimento.
df_full_ea = run_smap(P_ea, EP_ea, PARAMS_SG_CAL, COND_INICIAIS, AD_ea)
df_smap_ea = df_full_ea.iloc[WARMUP_MESES:]

Q_med_ea = df_smap_ea['Q_m3s'].mean()
Q_max_ea = df_smap_ea['Q_m3s'].max()
n_sec_ea = (df_smap_ea['Q_m3s'] < 0.001).sum()
print(f'Q_sim média  : {Q_med_ea:.4f} m³/s')
print(f'Q_sim máxima : {Q_max_ea:.4f} m³/s')
print(f'Meses secos  : {n_sec_ea}/{len(df_smap_ea)} ({100*n_sec_ea/len(df_smap_ea):.1f}%)')

# captura o nome do CSV exportado → consumido pela célula de download final
csv_ea = save_flow_csv(df_smap_ea, nome_ea)
print(f'CSV salvo    : {csv_ea}')

plot_simulation(df_smap_ea, nome_ea, PARAMS_SG_CAL)

---
## Download dos Arquivos Gerados

CSVs de vazões afluentes (formato largo, m³/s) e figuras PNG.

In [ ]:
from google.colab import files as colab_files

# CSVs de vazões afluentes — nomes capturados nas células de Simulação
colab_files.download(csv_ea)   # Eng. Ávidos (parâmetros transferidos de São Gonçalo)
colab_files.download(csv_sg)   # São Gonçalo (calibrado)

# Figuras PNG (descomente para baixar)
# colab_files.download('smap_engenheiro_avidos.png')
# colab_files.download('smap_sao_goncalo.png')
# colab_files.download('calibracao_sao_goncalo.png')
# colab_files.download('diagnostico_sao_goncalo.png')